# Run PLaTITO simulation starting from a structure from AlphaFold DB or PDB 

This notebook:
1. Downloads the structure.
2. Extracts ESMC 300M sequence embeddings via the EvolutionaryScale Forge API.
3. Loads the **PLaTITO-3M** checkpoint from HuggingFace and generates a trajectory.
4. Visualises the trajectory interactively and saves it as a movie.

**Before you start:** set your Forge API token in the cell below.

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

FORGE_TOKEN = os.environ.get("FORGE_TOKEN")

## Environment setup

Run the cell below **once** to verify your environment. If anything is missing it will print the install command.

```bash
# Create and activate the conda environment (only needed once):
mamba create -n platito python=3.11 -y
mamba activate platito

# Install PLaTITO and all dependencies:
pip install -e ".[cuda]" --extra-index-url https://download.pytorch.org/whl/cu126   # GPU
# pip install -e .                                                                  # CPU only

# Additional packages used by this notebook:
conda install -c conda-forge pymol-open-source -y
pip install biotite "imageio[ffmpeg]" ipympl nglview
```

In [2]:
import importlib.util, subprocess, sys

_required = {
    "torch":           "pip install torch",
    "lightning":       "pip install lightning",
    "hydra":           "pip install hydra-core",
    "mdtraj":          "pip install mdtraj",
    "biotite":         "pip install biotite",
    "imageio":         'pip install "imageio[ffmpeg]"',
    "ipywidgets":      "pip install ipywidgets",
    "ipympl":          "pip install ipympl",
    "huggingface_hub": "pip install huggingface_hub",
    "nglview":         "pip install nglview",
}

ok, missing = [], []
for pkg, cmd in _required.items():
    found = importlib.util.find_spec(pkg) is not None
    (ok if found else missing).append((pkg, cmd))

# pymol2 checked in a subprocess to avoid loading its native libs into this kernel
result = subprocess.run([sys.executable, "-c", "import pymol2"], capture_output=True)
pkg, cmd = "pymol2", "conda install -c conda-forge pymol-open-source"
(ok if result.returncode == 0 else missing).append((pkg, cmd))

print(f"Python  : {sys.version.split()[0]}")
print(f"OK      : {', '.join(p for p, _ in ok)}")
if missing:
    print("\nMISSING — run the following to install:")
    for pkg, cmd in missing:
        print(f"  {pkg:20s}  →  {cmd}")
else:
    print("All dependencies found — ready to run.")

Python  : 3.11.16
OK      : torch, lightning, hydra, mdtraj, biotite, imageio, ipywidgets, ipympl, huggingface_hub, nglview, pymol2
All dependencies found — ready to run.


## Configuration

In [3]:
# ── User-facing settings ──────────────────────────────────────────────────────

# Structure to simulate — two formats supported:
#   AF-<UniProt>-F1  →  downloaded from AlphaFold DB
#   <PDB ID>         →  downloaded from RCSB PDB

STRUCTURE_ID = "AF-P01542-F1"   # crambin, 46 residues (AFDB)

# PLaTITO checkpoint (HuggingFace Hub)
HF_REPO_ID  = "pantoniadis/platito"
HF_FILENAME = "platito_3M.ckpt"   # uses ESMC 300M embeddings

# Simulation settings
N_TRAJECTORIES  = 1      # parallel trajectories; increase to 1000 for publication-quality results
N_STEPS         = 100    # rollout steps per trajectory  → total = N_STEPS × LAG_NS ns
LAG_NS          = 10     # lag time Δt in ns
TEMPERATURE_K   = 320    # simulation temperature in Kelvin (training range: 320 / 348 / 379 / 413 / 450)
ODE_STEPS       = 50     # ODE integration steps per transition
ODE_METHOD      = "euler"

# Device: None → CPU
#         "mps" → Apple Silicon GPU
#         0, 1, … → CUDA GPU index
GPU_DEVICE      = 5

# Output directory derived from structure ID — no need to update manually
OUTPUT_DIR      = f"outputs/{STRUCTURE_ID}"
CHECKPOINT_DIR  = "checkpoints"          # local cache for HF model files

# Residue range for RMSD / RMSF (1-indexed, inclusive).
# Set to e.g. (7, 4) to exclude floppy termini; None = full sequence.
if STRUCTURE_ID == "AF-P01542-F1":
    RMSD_RESIDUES = (7,44)  # None
else:
    RMSD_RESIDUES = None

In [4]:
import os
import warnings
from pathlib import Path
import urllib.request

import torch
import numpy as np
import mdtraj as md
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

warnings.filterwarnings("ignore")

if isinstance(GPU_DEVICE, int):
    device = torch.device(f"cuda:{GPU_DEVICE}")
elif GPU_DEVICE == "mps":
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device : {device}")
print(f"PyTorch      : {torch.__version__}")
if device.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(GPU_DEVICE)}")
elif device.type == "mps":
    print(f"MPS available: {torch.backends.mps.is_available()}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

Using device : cuda:5
PyTorch      : 2.7.1+cu126
GPU          : NVIDIA H100 80GB HBM3


In [11]:
print([f"{x/2**30:.1f}GB" for x in torch.cuda.mem_get_info(device)])
a = torch.randn(64, 64, device=device); print((a @ a).sum())

RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Download the structure from AlphaFold DB or PDB

In [5]:
import json

pdb_path = os.path.join(OUTPUT_DIR, f"{STRUCTURE_ID}.pdb")

if not os.path.exists(pdb_path):
    if STRUCTURE_ID.startswith("AF-"):
        # AlphaFold DB — query API for the canonical versioned URL
        uniprot_id = STRUCTURE_ID.split("-")[1]
        api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
        with urllib.request.urlopen(api_url) as r:
            entry = json.loads(r.read())[0]
        pdb_url = entry["pdbUrl"]
        print(f"AlphaFold DB  : v{entry['latestVersion']}  {pdb_url}")
    else:
        # RCSB PDB
        pdb_url = f"https://files.rcsb.org/download/{STRUCTURE_ID}.pdb"
        print(f"RCSB PDB      : {pdb_url}")
    urllib.request.urlretrieve(pdb_url, pdb_path)
    print(f"Saved to {pdb_path}")
else:
    print(f"Found existing structure at {pdb_path}")


Found existing structure at outputs/AF-P01542-F1/AF-P01542-F1.pdb


In [6]:
from platito.utils.amino_acid_vocab import AA_1_TO_ID, AA_3_TO_1

traj_ref   = md.load(pdb_path)
ca_indices = traj_ref.topology.select("name CA")
ca_topo    = traj_ref.atom_slice(ca_indices).topology
ca_coords_start = traj_ref.xyz[0, ca_indices, :]  # [L, 3] in nm

sequence = "".join(
    AA_3_TO_1.get(traj_ref.topology.atom(i).residue.name, "X")
    for i in ca_indices
)
residue_ids = [AA_1_TO_ID.get(aa, AA_1_TO_ID["X"]) for aa in sequence]
L = len(sequence)

print(f"Structure     : {STRUCTURE_ID}")
print(f"Residues      : {L}")
print(f"Sequence      : {sequence}")

Structure     : AF-P01542-F1
Residues      : 46
Sequence      : TTCCPSIVARSNFNVCRLPGTPEALCATYTGCIIIPGATCPGDYAN


## Extract ESMC sequence embeddings

Embeddings are cached to disk; this cell is skipped on subsequent runs.

In [7]:
# Embedding filename depends on the checkpoint
if "6b" in HF_FILENAME or "19M" in HF_FILENAME or "36M" in HF_FILENAME:
    _esm_model  = "esmc-6b-2024-12"
    _emb_suffix = "esmc_6b"
else:
    _esm_model  = "esmc-300m-2024-12"
    _emb_suffix = "esmc_300m"

seq_emb_path = os.path.join(OUTPUT_DIR, f"seq_emb_{_emb_suffix}.pt")

if not os.path.exists(seq_emb_path):
    from esm.sdk.forge import ESM3ForgeInferenceClient
    from esm.sdk.api  import ESMProtein, LogitsConfig

    client = ESM3ForgeInferenceClient(
        model=_esm_model,
        url="https://forge.evolutionaryscale.ai",
        token=FORGE_TOKEN,
    )
    protein        = ESMProtein(sequence=sequence)
    protein_tensor = client.encode(protein)

    if _esm_model == "esmc-300m-2024-12":
        logits_output = client.logits(
            protein_tensor,
            LogitsConfig(sequence=True, return_hidden_states=True),
        )
        # Layer 20 hidden states, strip BOS/EOS → shape [L, 960]
        seq_emb = logits_output.hidden_states.squeeze(1)[20][1:-1, :].cpu()
    else:  # 6B
        logits_output = client.logits(
            protein_tensor,
            LogitsConfig(sequence=True, return_embeddings=True),
        )
        # Final embeddings, strip BOS/EOS → shape [L, 2560]
        seq_emb = logits_output.embeddings[0, 1:-1, :].cpu()

    assert seq_emb.shape[0] == L, f"Embedding length {seq_emb.shape[0]} != {L} residues"
    torch.save(seq_emb, seq_emb_path)
    print(f"Saved embeddings {tuple(seq_emb.shape)} → {seq_emb_path}")
else:
    seq_emb = torch.load(seq_emb_path, weights_only=False)
    print(f"Loaded cached embeddings {tuple(seq_emb.shape)} from {seq_emb_path}")

Loaded cached embeddings (46, 960) from outputs/AF-P01542-F1/seq_emb_esmc_300m.pt


## Load PLaTITO from HuggingFace

In [8]:
import lightning as pl
from huggingface_hub import hf_hub_download
from platito.models import PLaTITO

pl.seed_everything(12345, workers=True)
torch.set_float32_matmul_precision("medium")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
ckpt_path = os.path.join(CHECKPOINT_DIR, HF_FILENAME)
if os.path.exists(ckpt_path):
    print(f"Using cached checkpoint: {ckpt_path}")
else:
    print(f"Downloading {HF_REPO_ID}/{HF_FILENAME} …")
    hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME, local_dir=CHECKPOINT_DIR)
    print(f"Cached to {ckpt_path}")

model = PLaTITO.load_from_checkpoint(ckpt_path, map_location=device)
model.eval()
print("Model loaded.")

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters    : {total_params/1e6:.1f} M")

Seed set to 12345


Using cached checkpoint: checkpoints/platito_3M.ckpt
Model loaded.
Parameters    : 3.0 M


## Run PLaTITO simulation

In [9]:
# Build batched input (repeat the single starting structure N_TRAJECTORIES times)
x0 = (
    torch.tensor(ca_coords_start, dtype=torch.float32)
    .unsqueeze(0)
    .repeat(N_TRAJECTORIES, 1, 1)
)  # [N_TRAJECTORIES, L, 3]

seq_emb_batch = seq_emb.unsqueeze(0).repeat(N_TRAJECTORIES, 1, 1)  # [N_TRAJECTORIES, L, D]

rest_conditions = {
    "lag"         : torch.tensor([LAG_NS]       * N_TRAJECTORIES, dtype=torch.long).to(device),
    "temp"        : torch.tensor([TEMPERATURE_K] * N_TRAJECTORIES, dtype=torch.long).to(device),
    "residue_ids" : torch.tensor([residue_ids] * N_TRAJECTORIES, dtype=torch.long).to(device),
    "sequence_emb": seq_emb_batch.to(device),
}

total_time_ns = N_STEPS * LAG_NS
print(f"Simulating {N_TRAJECTORIES} trajectories using {N_STEPS} steps of {LAG_NS} ns = {total_time_ns} ns")

Simulating 1 trajectories using 100 steps of 10 ns = 1000 ns


In [10]:
generated_coords = model.generate_trajectory(
    x0=x0.to(device),
    trajectory_steps=N_STEPS,
    ode_steps=ODE_STEPS,
    ode_method=ODE_METHOD,
    return_intermediates=True,
    **rest_conditions,
)
# shape: [N_TRAJECTORIES, N_STEPS+1, L, 3] in nm

coords_path = os.path.join(OUTPUT_DIR, "generated_coords.pt")
torch.save(generated_coords.cpu(), coords_path)
print(f"Saved coordinates {tuple(generated_coords.shape)} → {coords_path}")

Generating trajectory:   0%|          | 0/100 [00:00<?, ?step/s]

Generating trajectory:   0%|          | 0/100 [00:00<?, ?step/s]


RuntimeError: CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasSgemm( handle, opa, opb, m, n, k, &alpha, a, lda, b, ldb, &beta, c, ldc)`

## Quick analysis: Cα RMSF and RMSD to start

In [ ]:
coords_np = generated_coords.cpu().numpy()  # [N_traj, N_steps+1, L, 3]

traj_analysis = md.Trajectory(coords_np[0], ca_topo)

# Residue sub-range for RMSD: converts 1-indexed inclusive range → 0-indexed atom indices.
if RMSD_RESIDUES is not None:
    res_start, res_end = RMSD_RESIDUES
    _rmsd_idx  = list(range(res_start - 1, res_end))
    _rmsd_desc = f"residues {res_start}–{res_end}"
else:
    _rmsd_idx  = None
    _rmsd_desc = "all residues"

# RMSD vs frame 0 over core residues
rmsd = md.rmsd(traj_analysis, traj_analysis, frame=0, atom_indices=_rmsd_idx)

# RMSF over the full sequence (superposition on all atoms)
rmsf = md.rmsf(traj_analysis, traj_analysis, frame=0)  # [L], nm

plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

res_axis = np.arange(1, L + 1)
axes[0].plot(res_axis, rmsf * 10, color="steelblue")
if RMSD_RESIDUES is not None:
    axes[0].axvspan(res_start, res_end, alpha=0.12, color="steelblue", label=f"RMSD core ({_rmsd_desc})")
    axes[0].legend(fontsize=8)
axes[0].set_xlabel("Residue")
axes[0].set_ylabel("RMSF (Å)")
axes[0].set_title(f"Cα RMSF — {STRUCTURE_ID} (full sequence)")

time_axis = np.arange(len(rmsd)) * LAG_NS
axes[1].plot(time_axis, rmsd * 10, color="tomato")
axes[1].set_xlabel("Time (ns)")
axes[1].set_ylabel("Cα RMSD from start (Å)")
axes[1].set_title(f"Trajectory RMSD — {STRUCTURE_ID} ({_rmsd_desc})")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "rmsf_rmsd.png"), dpi=150, bbox_inches="tight")
plt.show()


## Save trajectory to PDB / DCD for external viewers

In [ ]:
# Save Cα-only trajectory as PDB (readable by PyMOL, VMD, UCSF ChimeraX, etc.)
pdb_traj_path = os.path.join(OUTPUT_DIR, "trajectory_traj0_ca.pdb")
traj_analysis.save_pdb(pdb_traj_path)
print(f"Saved PDB trajectory ({len(traj_analysis)} frames) → {pdb_traj_path}")

# Also save as DCD (compact binary format)
dcd_path = os.path.join(OUTPUT_DIR, "trajectory_traj0_ca.dcd")
traj_analysis.save_dcd(dcd_path)
print(f"Saved DCD trajectory → {dcd_path}")

## PyMOL cartoon movie

Renders per-frame PyMOL cartoon ribbons using PULCHRA backbone reconstruction.
Secondary structure is assigned with biotite P-SEA and injected into PyMOL via `alter`.
Saves GIF and MOV to `paths.output_dir`.

In [ ]:
import importlib.util, subprocess, sys

_FPS = 12

_pymol_ok = (
    importlib.util.find_spec("pymol2") is not None
    and importlib.util.find_spec("biotite") is not None
)
if not _pymol_ok:
    print("pymol2 not found — skipping this cell.")
    print("  conda install -c conda-forge pymol-open-source -y")
else:
    import pymol2, imageio, tempfile
    import biotite.structure as struc
    from IPython.display import display

    if importlib.util.find_spec("pulchra") is None:
        print("Installing pulchra …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pulchra", "-q"])

    from pulchra.pulchra import Pulchra
    from pulchra.io.pdb_writer import write_pdb

    pymol_gif = os.path.join(OUTPUT_DIR, "trajectory_pymol.gif")
    pymol_mov = os.path.join(OUTPUT_DIR, "trajectory_pymol.mov")

    step_pymol = max(1, (N_STEPS + 1) // 100)
    pymol_xyz  = coords_np[0, ::step_pymol]   # [F, L, 3] nm
    n_frames   = len(pymol_xyz)

    _SS_MAP = {"a": "H", "b": "S", "c": "L"}

    _pulchra = Pulchra(
        verbose=False,
        ca_optimize=False,
        rebuild_backbone=True,
        rebuild_sidechains=False,
        optimize_xvol=False,
        optimize_hbonds=False,
    )

    def _make_ca_atoms(xyz_nm):
        n = len(xyz_nm)
        a = struc.AtomArray(n)
        a.coord        = xyz_nm * 10.0
        a.atom_name[:] = "CA"
        a.res_name[:]  = "ALA"
        a.chain_id[:]  = "A"
        a.res_id       = np.arange(1, n + 1)
        a.element[:]   = "C"
        return a

    def _rebuild_and_load(pm, xyz_nm, seq, sse, obj):
        pdb_tmp = os.path.join(tmpdir, f"{obj}.pdb")
        mol = _pulchra.reconstruct_coords(xyz_nm * 10.0, seq)
        write_pdb(pdb_tmp, mol)
        pm.cmd.load(pdb_tmp, obj)
        for resi, ss in zip(range(1, len(sse) + 1), sse):
            pm.cmd.alter(f"{obj} and resi {resi}", "ss=" + repr(_SS_MAP[ss]))
        pm.cmd.rebuild()
        pm.cmd.show("cartoon", obj)
        pm.cmd.color("red",    f"{obj} and ss h")
        pm.cmd.color("marine", f"{obj} and ss s")
        pm.cmd.color("gray70", f"{obj} and not (ss h or ss s)")

    tmpdir    = tempfile.mkdtemp()
    png_paths = []
    t_ns_list = []
    view      = None

    try:
        with pymol2.PyMOL() as pm:
            pm.cmd.bg_color("white")
            # load frame 0 as fixed reference for alignment
            _rebuild_and_load(pm, pymol_xyz[0], sequence, struc.annotate_sse(_make_ca_atoms(pymol_xyz[0])), "ref")
            pm.cmd.disable("ref")
            pm.cmd.orient("ref")
            pm.cmd.zoom("ref", 0.2)
            view = pm.cmd.get_view()

            for fi, xyz in enumerate(pymol_xyz):
                pm.cmd.bg_color("white")
                atoms = _make_ca_atoms(xyz)
                sse   = struc.annotate_sse(atoms)
                _rebuild_and_load(pm, xyz, sequence, sse, "fr")
                pm.cmd.align("fr", "ref")
                pm.cmd.set_view(view)
                png = os.path.join(tmpdir, f"f{fi:04d}.png")
                pm.cmd.png(png, width=640, height=480, ray=0, quiet=1)
                png_paths.append(png)
                t_ns_list.append(fi * step_pymol * LAG_NS)
                pm.cmd.delete("fr")
                print(f"  {fi+1:3d}/{n_frames}", end="\r", flush=True)
    except KeyboardInterrupt:
        print("Interrupted — saving rendered frames so far.")

    print(f"Rendered {len(png_paths)} frames")

    def _on_white(img):
        if img.ndim == 3 and img.shape[2] == 4:
            a = img[:,:,3:4].astype(float) / 255
            return (a * img[:,:,:3] + (1-a) * 255).astype(np.uint8)
        return img

    from PIL import Image, ImageDraw, ImageFont
    images = []
    for p, t_ns in zip(png_paths, t_ns_list):
        img = Image.fromarray(_on_white(imageio.imread(p)))
        draw = ImageDraw.Draw(img)
        label = f"{STRUCTURE_ID}   {t_ns:6d} ns"
        draw.text((12, img.height - 22), label, fill=(30, 30, 30))
        images.append(np.array(img))
    imageio.mimsave(pymol_gif, images, fps=_FPS, loop=0)
    print(f"Saved GIF → {pymol_gif}")
    try:
        imageio.mimsave(pymol_mov, images, fps=_FPS)
        print(f"Saved MOV → {pymol_mov}")
    except Exception as e:
        print(f"MOV skipped: {e}")

    for p in png_paths:
        try:
            os.remove(p)
        except OSError:
            pass

    from IPython.display import Image as IPImage
    display(IPImage(pymol_gif))


## Interactive 3D viewer

Interactive nglview widget with frame slider and mouse rotation.
Backbone is reconstructed with PULCHRA and cached to `trajectory_rebuilt.pdb`.

In [ ]:
import importlib.util, subprocess, sys, tempfile, os
import numpy as np
import mdtraj as md
import biotite.structure as struc
import nglview as nv

if importlib.util.find_spec("pulchra") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pulchra", "-q"])
from pulchra.pulchra import Pulchra
from pulchra.io.pdb_writer import write_pdb

rebuilt_pdb = os.path.join(OUTPUT_DIR, "trajectory_rebuilt.pdb")

if not os.path.exists(rebuilt_pdb):
    _pulchra = Pulchra(
        verbose=False, ca_optimize=False, rebuild_backbone=True,
        rebuild_sidechains=False, optimize_xvol=False, optimize_hbonds=False,
    )

    def _make_ca_atoms(xyz_nm):
        n = len(xyz_nm)
        a = struc.AtomArray(n)
        a.coord        = xyz_nm * 10.0
        a.atom_name[:] = "CA"
        a.res_name[:]  = "ALA"
        a.chain_id[:]  = "A"
        a.res_id       = np.arange(1, n + 1)
        a.element[:]   = "C"
        return a

    def _sse_to_pdb_records(sse):
        n = len(sse)
        records, helix_n, strand_n = [], 0, 0
        i = 0
        while i < n:
            ss = sse[i]
            if ss in ("a", "b"):
                j = i + 1
                while j < n and sse[j] == ss:
                    j += 1
                # trim residues that touch chain termini
                start = (i + 1) + (1 if i == 0 else 0)
                end   = j       - (1 if j == n else 0)
                length = end - start + 1
                if ss == "a" and length >= 3:
                    helix_n += 1
                    records.append(
                        f"HELIX  {helix_n:3d} {helix_n:3d} ALA A {start:4d}  ALA A {end:4d}  1{chr(32)*31}{length:5d}"
                    )
                elif ss == "b" and length >= 2:
                    strand_n += 1
                    records.append(
                        f"SHEET  {strand_n:3d}   A 1 ALA A {start:4d}  ALA A {end:4d}  0"
                    )
                i = j
            else:
                i += 1
        return records
        
    tmpdir = tempfile.mkdtemp()
    frame_trajs = []
    sse_frame0 = None

    for fi, xyz in enumerate(coords_np[0]):
        pdb_tmp = os.path.join(tmpdir, f"f{fi:04d}.pdb")
        mol = _pulchra.reconstruct_coords(xyz * 10.0, sequence)
        write_pdb(pdb_tmp, mol)
        frame_trajs.append(md.load(pdb_tmp))
        if fi == 0:
            sse_frame0 = struc.annotate_sse(_make_ca_atoms(xyz))
        print(f"  Rebuilding {fi+1}/{len(coords_np[0])}", end="\r", flush=True)

    combined = md.join(frame_trajs)
    combined.save_pdb(rebuilt_pdb)

    # Prepend HELIX/SHEET records (from frame 0 biotite P-SEA) before first MODEL
    ss_records = _sse_to_pdb_records(sse_frame0)
    with open(rebuilt_pdb, "r") as f:
        content = f.read()
    insert = content.find("MODEL")
    content = content[:insert] + "\n".join(ss_records) + "\n" + content[insert:]
    with open(rebuilt_pdb, "w") as f:
        f.write(content)
    print(f"Saved {rebuilt_pdb}  ({len(ss_records)} SS records)")
else:
    print(f"Using cached: {rebuilt_pdb}")

traj = md.load(rebuilt_pdb)
view = nv.NGLWidget()
comp = view.add_trajectory(nv.MDTrajTrajectory(traj), default_representation=False)
# Single representation — NGL needs full chain context for strand arrows
comp.add_cartoon(color_scheme="sstruc")
view.center()
view


In [ ]:
from pathlib import Path

print(f"Output directory: {OUTPUT_DIR}\n")
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f"  {f.name:45s} {f.stat().st_size / 1e6:.1f} MB")
